# Day 17: SQL 字符串函数与日期处理 —— 习题

> **范围**: 字符串函数(UPPER/LOWER/TRIM/LENGTH/SUBSTR/REPLACE/CONCAT/LIKE) + 日期函数(STRFTIME/EXTRACT/DATE_DIFF/DATE_TRUNC)
> **数据**: `../data/sales.csv` + `../data/customers.csv`
> **建议用时**: 60-90 分钟
> **注意**: DuckDB 的日期函数语法，CSV 读入的日期默认是字符串

In [2]:
import duckdb

%load_ext sql
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

## Easy

**1. 字符串大小写与去空格**

基于 `../data/sales.csv`：
- 用 SQL 把 `country` 列统一转为大写，去重显示所有国家
- 用 `TRIM` 清理 `product` 列（假设可能有空格），显示前5个去重后的产品名
- 计算每个 `product` 名字的长度，显示前5个最长的产品名

In [ ]:
%%sql
SELECT DISTINCT UPPER(country) AS country_upper
FROM '../data/sales.csv';

Running query in 'duckdb:///:memory:'

country_upper
FRANCE
CHINA
UK
GERMANY
US


In [5]:
%%sql
SELECT DISTINCT TRIM(product) AS product_trimmed
FROM '../data/sales.csv'
LIMIT 5;

Running query in 'duckdb:///:memory:'

product_trimmed
Monitor
Headphones
Laptop
Phone
Keyboard


In [7]:
%%sql
SELECT DISTINCT product,
       LENGTH(product) AS product_length
FROM '../data/sales.csv'
ORDER BY product_length DESC
LIMIT 5;

Running query in 'duckdb:///:memory:'

product,product_length
Headphones,10
Keyboard,8
Monitor,7
Laptop,6
Phone,5


**2. 子串提取与类型转换**

基于 `../data/sales.csv`：
- 提取 `order_id` 的数字部分（去掉首字母 'O'），转为整数，显示前5行
  （提示：`SUBSTR(order_id, 2, 10)` + `CAST(... AS INTEGER)`）
- 提取 `customer_id` 的第3个字符（如 'C001' → '0'），显示前5行
- 统计 `order_id` 数字部分 > 1200 的订单数量

In [ ]:
%%sql
SELECT order_id,
       CAST(SUBSTR(order_id, 2, 10) AS INTEGER) AS order_num_int
FROM '../data/sales.csv'
LIMIT 5;   

Running query in 'duckdb:///:memory:'

order_id,order_num_int
O1000,1000
O1001,1001
O1002,1002
O1003,1003
O1004,1004


In [12]:
%%sql
SELECT customer_id, 
       SUBSTR(customer_id, 3, 1) AS customer_num
FROM '../data/sales.csv'
LIMIT 5;

Running query in 'duckdb:///:memory:'

customer_id,customer_num
C007,0
C004,0
C005,0
C007,0
C003,0


In [10]:
%%sql
SELECT order_id,
       CAST(SUBSTR(order_id, 2, 10) AS INTEGER) AS order_num_int 
FROM '../data/sales.csv'
WHERE order_num_int > 1200;

Running query in 'duckdb:///:memory:'

order_id,order_num_int
O1201,1201
O1202,1202
O1203,1203
O1204,1204
O1205,1205
O1206,1206
O1207,1207
O1208,1208
O1209,1209
O1210,1210


**3. 日期提取与格式化**

基于 `../data/sales.csv`：
- 用 `STRFTIME` 提取 `order_date` 的年月格式 `YYYY-MM`，显示前5行
- 用 `EXTRACT` 提取 `order_date` 的年份和月份，分别显示
- 按 `YYYY-MM` 分组统计每个月的订单数量，按月份排序

In [14]:
%%sql
SELECT order_date,
       STRFTIME(order_date, '%Y-%m') AS order_date_year_month,
       EXTRACT(YEAR FROM order_date) AS order_year,
       EXTRACT(MONTH FROM order_date) AS order_month
FROM '../data/sales.csv'
LIMIT 5;       

Running query in 'duckdb:///:memory:'

order_date,order_date_year_month,order_year,order_month
2024-01-01,2024-01,2024,1
2024-01-01,2024-01,2024,1
2024-01-02,2024-01,2024,1
2024-01-03,2024-01,2024,1
2024-01-03,2024-01,2024,1


In [15]:
%%sql
SELECT STRFTIME('%Y-%m', order_date) AS order_date_year_month,
       COUNT(*) AS order_count
FROM '../data/sales.csv'
GROUP BY order_date_year_month
ORDER BY order_date_year_month;


Running query in 'duckdb:///:memory:'

order_date_year_month,order_count
2024-01,43
2024-02,40
2024-03,42
2024-04,41
2024-05,42
2024-06,41
2024-07,43
2024-08,42
2024-09,41
2024-10,42


## Medium

**4. 字符串替换与拼接**

基于 `../data/sales.csv` 和 `../data/customers.csv`：
- 把 `sales` 表中 `country` 为 `"US"` 的替换为 `"USA"`，显示替换后的前5行（country 和 order_id）
- 拼接 `customer_id` 和 `country` 为一个标签列 `"customer_id_country"`，格式如 `"C001_US"`，显示前5行
- LEFT JOIN `sales` 和 `customers`，拼接 `name` 和 `country` 为 `"name (country)"` 格式（如 `"Alice (France)"`），显示前5行

In [16]:
%%sql
SELECT REPLACE(country, 'US', 'USA') AS country_replaced,
       order_id
FROM '../data/sales.csv'
LIMIT 5;

Running query in 'duckdb:///:memory:'

country_replaced,order_id
Germany,O1000
USA,O1001
USA,O1002
USA,O1003
France,O1004


In [17]:
%%sql
SELECT customer_id, country,
       CONCAT(customer_id, '_', country) AS customer_id_country
FROM '../data/sales.csv'
LIMIT 5;

Running query in 'duckdb:///:memory:'

customer_id,country,customer_id_country
C007,Germany,C007_Germany
C004,US,C004_US
C005,US,C005_US
C007,US,C007_US
C003,France,C003_France


In [18]:
%%sql
SELECT s.*,
       CONCAT(c.name, ' (', c.country, ')') AS customer_info
FROM '../data/sales.csv' AS s
LEFT JOIN '../data/customers.csv' AS c
    ON s.customer_id = c.customer_id
LIMIT 5;

Running query in 'duckdb:///:memory:'

order_id,customer_id,product,category,quantity,price,order_date,country,total,customer_info
O1000,C007,Keyboard,Accessory,2,1299,2024-01-01,Germany,2598,Grace (Germany)
O1001,C004,Keyboard,Accessory,1,99,2024-01-01,US,99,David (US)
O1002,C005,Laptop,Computer,4,99,2024-01-02,US,396,Eva (US)
O1003,C007,Headphones,Audio,4,99,2024-01-03,US,396,Grace (Germany)
O1004,C003,Phone,Mobile,5,99,2024-01-03,France,495,Charlie (France)


**5. 模糊匹配与筛选**

基于 `../data/sales.csv`：
- 用 `LIKE` 筛选 `product` 以 `"Key"` 开头的产品，统计数量和总销售额
- 用 `ILIKE` 筛选 `product` 包含 `"phone"`（不区分大小写）的订单，显示前5行
- 用 `LENGTH` 筛选 `product` 名字长度 >= 8 的订单，统计数量

In [21]:
%%sql
SELECT product,
       COUNT(*) AS product_count,
       SUM(total) AS total_sales
FROM '../data/sales.csv'
WHERE product LIKE '%Key%'
GROUP BY product;

Running query in 'duckdb:///:memory:'

product,product_count,total_sales
Keyboard,99,225111


In [22]:
%%sql
SELECT *
FROM '../data/sales.csv'
WHERE product ILIKE '%phone%'
LIMIT 5;

Running query in 'duckdb:///:memory:'

order_id,customer_id,product,category,quantity,price,order_date,country,total
O1003,C007,Headphones,Audio,4,99,2024-01-03,US,396
O1004,C003,Phone,Mobile,5,99,2024-01-03,France,495
O1006,C005,Phone,Mobile,5,1299,2024-01-05,UK,6495
O1008,C007,Phone,Mobile,5,299,2024-01-06,UK,1495
O1009,C002,Headphones,Audio,3,99,2024-01-07,Germany,297


In [24]:
%%sql
SELECT product,
       COUNT(*) AS product_count
FROM '../data/sales.csv'
WHERE LENGTH(product) >= 8
GROUP BY product


Running query in 'duckdb:///:memory:'

product,product_count
Keyboard,99
Headphones,67


**6. 日期差与分组**

基于 `../data/sales.csv`：
- 计算每个订单与 `2024-06-01` 的日期差（天数），显示 order_id 和天数，前5行
  （提示：`DATE_DIFF('day', order_date, CAST('2024-06-01' AS DATE))`）
- 按 `DATE_TRUNC('month', order_date)` 分组，统计每个月的订单数和总销售额
- 找出距离 `2024-06-01` 最近的 5 个订单（按绝对差排序）
  （提示：`ABS(DATE_DIFF(...))`）

In [27]:
%%sql
SELECT order_id,
       order_date,
       DATE_DIFF('day', order_date, CAST('2024-06-01' AS DATE)) AS days_ago
FROM '../data/sales.csv'
LIMIT 5;

Running query in 'duckdb:///:memory:'

order_id,order_date,days_ago
O1000,2024-01-01,152
O1001,2024-01-01,152
O1002,2024-01-02,151
O1003,2024-01-03,150
O1004,2024-01-03,150


In [30]:
%%sql
SELECT DATE_TRUNC('month', order_date) AS order_month,
       COUNT(*) AS order_count,
       SUM(total) AS total_sales
FROM '../data/sales.csv'
GROUP BY order_month
ORDER BY order_month;

Running query in 'duckdb:///:memory:'

order_month,order_count,total_sales
2024-01-01 00:00:00,43,99365
2024-02-01 00:00:00,40,81795
2024-03-01 00:00:00,42,133371
2024-04-01 00:00:00,41,123882
2024-05-01 00:00:00,42,71176
2024-06-01 00:00:00,41,119061
2024-07-01 00:00:00,43,101271
2024-08-01 00:00:00,42,104388
2024-09-01 00:00:00,41,103183
2024-10-01 00:00:00,42,117265


In [31]:
%%sql
SELECT
    *,
    ABS(DATE_DIFF('day', order_date, '2024-06-01')) AS day_diff_abs
FROM '../data/sales.csv'
ORDER BY day_diff_abs ASC
LIMIT 5;

Running query in 'duckdb:///:memory:'

order_id,customer_id,product,category,quantity,price,order_date,country,total,day_diff_abs
O1209,C005,Mouse,Accessory,3,1299,2024-06-01,UK,3897,0
O1208,C006,Headphones,Audio,3,999,2024-06-01,Germany,2997,0
O1210,C007,Phone,Mobile,3,99,2024-06-02,UK,297,1
O1207,C004,Laptop,Computer,3,599,2024-05-31,UK,1797,1
O1206,C008,Headphones,Audio,5,599,2024-05-30,Germany,2995,2


**7. 字符串清洗 + 日期分组**

基于 `../data/sales.csv`：
- 用 `TRIM(LOWER(product))` 清洗产品名，存为 `product_clean`（用计算列）
- 按 `product_clean` 和 `STRFTIME('%Y-%m', order_date)` 分组，统计每个产品每月的订单数
- 只保留订单数 >= 5 的组（用 `HAVING`）
- 按年月升序、订单数降序排列

In [34]:
%%sql
SELECT
       TRIM(LOWER(product)) AS product_cleaned,
        STRFTIME('%Y-%m', order_date) AS order_year_month,
       COUNT(*) AS order_count
FROM '../data/sales.csv'
GROUP BY product_cleaned, order_year_month
HAVING COUNT(*) >= 5
ORDER BY order_year_month, order_count DESC;

Running query in 'duckdb:///:memory:'

product_cleaned,order_year_month,order_count
phone,2024-01,13
laptop,2024-01,8
headphones,2024-01,6
monitor,2024-01,6
mouse,2024-01,5
keyboard,2024-01,5
mouse,2024-02,9
monitor,2024-02,9
phone,2024-02,7
keyboard,2024-02,6


## Hard

**8. 综合字符串分析 —— 客户名处理**

基于 `../data/customers.csv`：
- 统计每个客户名 `name` 的长度，找出最长和最短的名字
- 用 `SUBSTR` 提取 `name` 的首字母，统计每个首字母的客户数量
- 用 `CONCAT` 拼接 `name` 和 `signup_date`，格式如 `"Alice_2023-01-15"`
- LEFT JOIN `sales` 和 `customers`，统计每个客户（`customer_id` + `name`）的首字母和总消费金额

In [38]:
%%sql
SELECT
    customer_id,
    name,
    LENGTH(name) AS name_length
FROM '../data/customers.csv'
ORDER BY name_length DESC;

Running query in 'duckdb:///:memory:'

customer_id,name,name_length
C003,Charlie,7
C001,Alice,5
C004,David,5
C006,Frank,5
C007,Grace,5
C008,Henry,5
C010,Jack,4
C011,Kate,4
C002,Bob,3
C005,Eva,3


In [ ]:
%%sql
SELECT
    name,
    LENGTH(name) AS max_name_length
FROM '../data/customers.csv'
WHERE LENGTH(name) = (SELECT MAX(LENGTH(name)) FROM '../data/customers.csv');

Running query in 'duckdb:///:memory:'

name,max_name_length
Charlie,7


In [41]:
%%sql
SELECT
    name,
    LENGTH(name) AS min_name_length
FROM '../data/customers.csv'
WHERE LENGTH(name) = (SELECT MIN(LENGTH(name)) FROM '../data/customers.csv');

Running query in 'duckdb:///:memory:'

name,min_name_length
Bob,3
Eva,3
Ivy,3


In [44]:
%%sql
SELECT 
       UPPER(SUBSTR(name, 1, 1)) AS name_initial_upper,
       COUNT(*) AS customer_count
FROM '../data/customers.csv'
GROUP BY name_initial_upper
ORDER BY name_initial_upper ASC;

Running query in 'duckdb:///:memory:'

name_initial_upper,customer_count
A,1
B,1
C,1
D,1
E,1
F,1
G,1
H,1
I,1
J,1


In [45]:
%%sql
SELECT
    customer_id,
    name,
    signup_date,
    CONCAT(name, '_', signup_date) AS name_signup_key
FROM '../data/customers.csv'
LIMIT 10;

Running query in 'duckdb:///:memory:'

customer_id,name,signup_date,name_signup_key
C001,Alice,2023-01-15,Alice_2023-01-15
C002,Bob,2023-02-20,Bob_2023-02-20
C003,Charlie,2023-03-10,Charlie_2023-03-10
C004,David,2023-04-05,David_2023-04-05
C005,Eva,2023-05-12,Eva_2023-05-12
C006,Frank,2023-06-18,Frank_2023-06-18
C007,Grace,2023-07-22,Grace_2023-07-22
C008,Henry,2023-08-30,Henry_2023-08-30
C009,Ivy,2023-09-14,Ivy_2023-09-14
C010,Jack,2023-10-01,Jack_2023-10-01


In [48]:
%%sql
SELECT
    c.customer_id,
    c.name,
    UPPER(SUBSTR(c.name, 1, 1)) AS first_letter,
    SUM(s.total) AS total_spent
FROM '../data/customers.csv' c
LEFT JOIN '../data/sales.csv' s
    ON c.customer_id = s.customer_id
GROUP BY
    c.customer_id,
    c.name,
    first_letter
ORDER BY total_spent DESC;

Running query in 'duckdb:///:memory:'

customer_id,name,first_letter,total_spent
C006,Frank,F,198806
C001,Alice,A,184001
C008,Henry,H,164484
C003,Charlie,C,160212
C004,David,D,144095
C007,Grace,G,143917
C002,Bob,B,140967
C005,Eva,E,131234
C009,Ivy,I,None
C011,Kate,K,None


**9. 日期分析 —— 季度与周末**

基于 `../data/sales.csv`：
- 创建「季度」列：用 `CASE` 根据月份判断 Q1/Q2/Q3/Q4
  ```sql
  CASE
    WHEN EXTRACT(MONTH FROM order_date) <= 3 THEN 'Q1'
    WHEN EXTRACT(MONTH FROM order_date) <= 6 THEN 'Q2'
    WHEN EXTRACT(MONTH FROM order_date) <= 9 THEN 'Q3'
    ELSE 'Q4'
  END AS quarter
  ```
- 按季度统计订单数和总销售额
- 用 `EXTRACT(DOW FROM order_date)` 判断星期几（0=周日，1=周一...），统计每天下单数量
- 找出下单最多的星期几

In [49]:
%%sql
SELECT
    order_id,
    order_date,
    CASE
        WHEN EXTRACT(MONTH FROM order_date) <= 3 THEN 'Q1'
        WHEN EXTRACT(MONTH FROM order_date) <= 6 THEN 'Q2'
        WHEN EXTRACT(MONTH FROM order_date) <= 9 THEN 'Q3'
        ELSE 'Q4'
    END AS quarter,
    EXTRACT(DOW FROM order_date) AS dow_num
FROM '../data/sales.csv'
LIMIT 10;


Running query in 'duckdb:///:memory:'

order_id,order_date,quarter,dow_num
O1000,2024-01-01,Q1,1
O1001,2024-01-01,Q1,1
O1002,2024-01-02,Q1,2
O1003,2024-01-03,Q1,3
O1004,2024-01-03,Q1,3
O1005,2024-01-04,Q1,4
O1006,2024-01-05,Q1,5
O1007,2024-01-06,Q1,6
O1008,2024-01-06,Q1,6
O1009,2024-01-07,Q1,0


In [50]:
%%sql
SELECT
    CASE
        WHEN EXTRACT(MONTH FROM order_date) <= 3 THEN 'Q1'
        WHEN EXTRACT(MONTH FROM order_date) <= 6 THEN 'Q2'
        WHEN EXTRACT(MONTH FROM order_date) <= 9 THEN 'Q3'
        ELSE 'Q4'
    END AS quarter,
    COUNT(*) AS total_orders, 
    SUM(total) AS total_sales               
FROM '../data/sales.csv'
GROUP BY quarter
ORDER BY quarter;

Running query in 'duckdb:///:memory:'

quarter,total_orders,total_sales
Q1,125,314531
Q2,124,314119
Q3,126,308842
Q4,125,330224


In [51]:
%%sql
SELECT
    EXTRACT(DOW FROM order_date) AS dow_num,
    COUNT(*) AS order_count
FROM '../data/sales.csv'
GROUP BY dow_num
ORDER BY dow_num;

Running query in 'duckdb:///:memory:'

dow_num,order_count
0,69
1,75
2,70
3,73
4,70
5,70
6,73


In [53]:
%%sql
SELECT
    EXTRACT(DOW FROM order_date) AS dow_num,
    COUNT(*) AS order_count
FROM '../data/sales.csv'
GROUP BY dow_num
ORDER BY order_count DESC
LIMIT 1;

Running query in 'duckdb:///:memory:'

dow_num,order_count
1,75


**10. 综合管道 —— 清洗 + 关联 + 分组 + 日期**

写一段完整 SQL 脚本（可以分多个 cell）：

**阶段1 —— 清洗（用 CTE）**:
```sql
WITH clean_sales AS (
  SELECT
    order_id,
    customer_id,
    TRIM(LOWER(product)) AS product_clean,
    category,
    quantity,
    total,
    order_date,
    UPPER(country) AS country_upper,
    STRFTIME('%Y-%m', order_date) AS ym
  FROM '../data/sales.csv'
)
```

**阶段2 —— 关联**:
- LEFT JOIN `clean_sales` 和 `customers`（`customer_id`）

**阶段3 —— 分组**:
- 按 `country_upper` 和 `ym` 分组，统计订单数和总销售额
- 按 `name` 分组，统计每个客户的总消费、订单数、最后下单日期（用 `MAX(order_date)`）

**阶段4 —— 输出**:
- 返回两个结果集（可以分成两个查询 cell）：
  1. 国家-月份统计
  2. 客户统计（包含最后下单日期）

完全没懂这个CTE是什么。

参考答案

In [3]:
%%sql
-- 阶段1+2+3: 国家-月份统计
WITH clean_sales AS (
  SELECT
    order_id,
    customer_id,
    TRIM(LOWER(product)) AS product_clean,
    category,
    quantity,
    total,
    order_date,
    UPPER(country) AS country_upper,
    STRFTIME('%Y-%m', order_date) AS ym
  FROM '../data/sales.csv'
)
SELECT 
    c.country_upper,
    c.ym,
    COUNT(*) AS order_count,
    SUM(c.total) AS total_sales
FROM clean_sales c
LEFT JOIN '../data/customers.csv' cust
    ON c.customer_id = cust.customer_id
GROUP BY c.country_upper, c.ym
ORDER BY c.ym, total_sales DESC;

Running query in 'duckdb:///:memory:'

country_upper,ym,order_count,total_sales
UK,2024-01,15,38553
US,2024-01,10,19370
FRANCE,2024-01,9,17974
CHINA,2024-01,4,14284
GERMANY,2024-01,5,9184
UK,2024-02,17,48649
US,2024-02,10,18575
GERMANY,2024-02,6,7388
FRANCE,2024-02,4,5393
CHINA,2024-02,3,1790


In [4]:
%%sql
-- 阶段4: 客户统计
WITH clean_sales AS (
  SELECT
    order_id,
    customer_id,
    total,
    order_date,
    STRFTIME('%Y-%m', order_date) AS ym
  FROM '../data/sales.csv'
)
SELECT 
    cust.name,
    COUNT(*) AS order_count,
    SUM(c.total) AS total_spent,
    MAX(c.order_date) AS last_order_date
FROM clean_sales c
LEFT JOIN '../data/customers.csv' cust
    ON c.customer_id = cust.customer_id
GROUP BY cust.name
ORDER BY total_spent DESC;

Running query in 'duckdb:///:memory:'

name,order_count,total_spent,last_order_date
Frank,62,198806,2024-12-26
Alice,71,184001,2024-12-30
Henry,72,164484,2024-12-24
Charlie,64,160212,2024-12-28
David,76,144095,2024-12-31
Grace,58,143917,2024-12-28
Bob,46,140967,2024-12-02
Eva,51,131234,2024-12-27


CTE = Common Table Expression = 临时命名子查询

In [ ]:
# -- 语法：WITH 名字 AS (子查询)
# WITH 清洗后的数据 AS (
#   SELECT TRIM(LOWER(product)) AS product_clean, total
#   FROM '../data/sales.csv'
# )
# SELECT product_clean, SUM(total) FROM 清洗后的数据 GROUP BY product_clean;

类比 Python:

In [ ]:
# 先赋值一个中间变量
# clean_data = df.copy()
# clean_data['product'] = df['product'].str.lower().str.strip()

# 再用这个中间变量做分析
# result = clean_data.groupby('product')['total'].sum()

CTE 就是 SQL 里的「中间变量」，让复杂查询可以分步写。